# Quantization trade-offs: fp16 vs int8 vs int4

Compare-it lab (LAB-CREATION-BLUEPRINT Section 21). Three competing architectures: fp16 baseline, int8 via bitsandbytes, int4 via AWQ. You stand up all three Modal L4 endpoints in parallel, benchmark latency / throughput / GPU memory / accuracy against a 50-example eval, and capture a three-way comparison metric. The validator NEVER decides which precision wins; that lives in the reflection cell.

**Duration:** about 120 minutes of work in a 150-minute session window.

**Cost preamble.** Three Modal L4 endpoints in parallel. About five bucks on a clean run. The most expensive Compare-it lab in the track. Three watchdogs are your safety net.

**CLEANUP IS CRITICAL.** A forgotten Modal L4 burns ~$26/day; three forgotten L4s burn ~$78/day. The cleanup cell tears down ALL THREE apps + Volumes. Run it before you walk away.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies.

!pip install --quiet labrun-checks==0.7.0 modal==0.66.40 anthropic==0.42.0 huggingface-hub==0.26.0 requests==2.32.0 numpy==1.26.4

In [ ]:
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import statistics
import sys
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed

import modal
import numpy as np
import requests

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "quantization-tradeoffs-fp16-int8-int4"
LAB_TAG_VALUE = LAB_ID

APP_NAMES = {
    "fp16": f"labrun-{LAB_ID}-app-fp16",
    "int8": f"labrun-{LAB_ID}-app-int8",
    "int4": f"labrun-{LAB_ID}-app-int4",
}
VOLUME_NAMES = {p: f"labrun-{LAB_ID}-weights-{p}" for p in APP_NAMES}

BENCHMARK_PATH = "/content/quantization_benchmark.json"

ANTHROPIC_SONNET = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

# Model ids per precision. int4 uses an AWQ-quantized HuggingFace checkpoint.
MODEL_IDS = {
    "fp16": "mistralai/Mistral-7B-Instruct-v0.3",
    "int8": "mistralai/Mistral-7B-Instruct-v0.3",  # int8 is applied at load time via bitsandbytes
    "int4": "TheBloke/Mistral-7B-Instruct-v0.2-AWQ",  # pre-quantized AWQ
}

In [ ]:
# NBVAL_SKIP
# BYOK setup.

import anthropic

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
MODAL_TOKEN_ID = getpass.getpass("MODAL_TOKEN_ID: ").strip()
MODAL_TOKEN_SECRET = getpass.getpass("MODAL_TOKEN_SECRET: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
HF_TOKEN = getpass.getpass("HF_TOKEN (optional): ").strip()

if not all([MODAL_TOKEN_ID, MODAL_TOKEN_SECRET, ANTHROPIC_API_KEY]):
    print("Modal + Anthropic credentials are required. Re-run.")
    raise SystemExit(1)

os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(model=ANTHROPIC_HAIKU, max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}])
    print(f"Anthropic OK. Haiku said: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic rejected: {exc!r}")
    raise SystemExit(1)

try:
    modal.App.lookup("labrun-nonexistent-probe-app", create_if_missing=False)
except modal.exception.NotFoundError:
    print("Modal credentials validated.")
except Exception as exc:
    print(f"Modal probe: {exc!r}. Treat as auth failure if persistent.")

print("STARTING THREE L4 ENDPOINTS IN PARALLEL.")
print("Cost: $3.30/hour while all three are warm.")
print("CLEANUP IS MANDATORY at end of lab.")

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"app_names": APP_NAMES},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit, orphan scan.
# Order: int4 app first, then int8, then fp16 (reverse-creation). Volumes after apps.

CLEANUP_MANIFEST = [
    CleanupResource(type="local_file", id=BENCHMARK_PATH, region="local",
        cli_delete_command=f"rm -f {BENCHMARK_PATH}"),
    CleanupResource(type="modal_volume", id=VOLUME_NAMES["int4"], region="modal",
        cli_delete_command=f"modal volume delete {VOLUME_NAMES['int4']}"),
    CleanupResource(type="modal_volume", id=VOLUME_NAMES["int8"], region="modal",
        cli_delete_command=f"modal volume delete {VOLUME_NAMES['int8']}"),
    CleanupResource(type="modal_volume", id=VOLUME_NAMES["fp16"], region="modal",
        cli_delete_command=f"modal volume delete {VOLUME_NAMES['fp16']}"),
    CleanupResource(type="modal_app", id=APP_NAMES["int4"], region="modal",
        cli_delete_command=f"modal app stop {APP_NAMES['int4']} && modal app delete {APP_NAMES['int4']}"),
    CleanupResource(type="modal_app", id=APP_NAMES["int8"], region="modal",
        cli_delete_command=f"modal app stop {APP_NAMES['int8']} && modal app delete {APP_NAMES['int8']}"),
    CleanupResource(type="modal_app", id=APP_NAMES["fp16"], region="modal",
        cli_delete_command=f"modal app stop {APP_NAMES['fp16']} && modal app delete {APP_NAMES['fp16']}"),
]


class QuantizationLabCleanupAdapter:
    """Modal app + volume + local. Three apps; all critical.

    TODO: labrun-checks 0.8 ships Modal adapter.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "modal_app":
            try:
                app = modal.App.lookup(resource.id, create_if_missing=False)
                if hasattr(app, "stop"):
                    try: app.stop()
                    except Exception: pass
                if hasattr(modal.App, "delete"):
                    try: modal.App.delete(resource.id)
                    except Exception: pass
            except modal.exception.NotFoundError:
                return
            except Exception:
                return
            return
        if resource.type == "modal_volume":
            try:
                vol = modal.Volume.from_name(resource.id)
                if hasattr(vol, "delete"):
                    vol.delete()
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "modal_app":
            try:
                modal.App.lookup(resource.id, create_if_missing=False)
                return True
            except Exception:
                return False
        if resource.type == "modal_volume":
            try:
                modal.Volume.from_name(resource.id)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        return []


CLEANUP_ADAPTER = QuantizationLabCleanupAdapter()


_orphans = []
if os.path.exists(BENCHMARK_PATH):
    _orphans.append(BENCHMARK_PATH)
for n in APP_NAMES.values():
    try:
        modal.App.lookup(n, create_if_missing=False)
        _orphans.append(f"modal://{n}")
    except Exception:
        pass

if _orphans:
    print("Orphan scan found leftover state from a prior session:")
    for o in _orphans:
        print(f"  {o}")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Task 1: Deploy all three quantized endpoints

Three Modal apps. Each loads Mistral 7B with a different precision:

- **fp16:** plain `vllm.LLM` (default dtype is float16 on L4).
- **int8:** `vllm.LLM(quantization="bitsandbytes", load_format="bitsandbytes")`.
- **int4:** load `TheBloke/Mistral-7B-Instruct-v0.2-AWQ` (pre-quantized).

Each captures GPU memory in MB inside the container via `torch.cuda.memory_allocated()` and surfaces it on the response.

The validator confirms all three endpoints return 200 with a `text` field AND a `gpu_memory_mb` field. Comparison ("which uses less memory") is NOT validated here per Compare-it rules.

In [ ]:
# Task 1: define three Modal apps.

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("vllm==0.6.3", "transformers==4.45.2", "bitsandbytes==0.44.1", "autoawq==0.2.6")
)


def make_app(precision):
    """Build one Modal app for a precision tier."""
    app_name = APP_NAMES[precision]
    vol_name = VOLUME_NAMES[precision]
    model_id = MODEL_IDS[precision]
    volume = modal.Volume.from_name(vol_name, create_if_missing=True)
    app = modal.App(app_name)

    @app.cls(
        gpu="L4",
        image=vllm_image,
        volumes={"/weights": volume},
        min_containers=1,
        max_containers=1,
        scaledown_window=180,
        timeout=900,
    )
    class _Server:
        @modal.enter()
        def load_model(self):
            from vllm import LLM
            import torch
            kwargs = {"model": model_id, "max_model_len": 4096, "download_dir": "/weights"}
            # YOUR CODE: branch on precision. For "int8", add
            # quantization="bitsandbytes", load_format="bitsandbytes",
            # gpu_memory_utilization=0.75 (int8 saves some VRAM but vLLM still
            # needs headroom for the KV cache). For "int4", pass quantization="awq"
            # only; the AWQ checkpoint is already quantized; gpu_memory_utilization=0.70.
            # For "fp16", just use the defaults.
            raise NotImplementedError("Replace with per-precision kwargs.")
            self.llm = LLM(**kwargs)
            torch.cuda.synchronize()
            self.gpu_memory_mb = int(torch.cuda.memory_allocated() / (1024 * 1024))
            print(f"Loaded {precision}; gpu_memory_mb={self.gpu_memory_mb}")

        @modal.method()
        def generate(self, prompt, max_tokens=64):
            from vllm import SamplingParams
            t0 = time.perf_counter()
            outputs = self.llm.generate([prompt], SamplingParams(max_tokens=max_tokens, temperature=0.0))
            latency_ms = int((time.perf_counter() - t0) * 1000)
            text = outputs[0].outputs[0].text
            return {"text": text, "latency_ms": latency_ms, "gpu_memory_mb": self.gpu_memory_mb}

    @app.function(image=vllm_image, timeout=120)
    @modal.web_endpoint(method="POST", label=f"{precision}-generate")
    def endpoint(payload: dict):
        result = _Server().generate.remote(payload["prompt"], max_tokens=payload.get("max_tokens", 64))
        return result

    return app, endpoint


# Deploy all three in parallel via thread pool (Modal SDK is sync per call but
# parallelism here saves 3-5 minutes on cold-load).
APPS = {}
ENDPOINTS = {}


def deploy_one(precision):
    app, endpoint = make_app(precision)
    app.deploy()
    return precision, app, endpoint


with ThreadPoolExecutor(max_workers=3) as ex:
    futs = {ex.submit(deploy_one, p): p for p in APP_NAMES}
    for fut in as_completed(futs):
        precision, app, endpoint = fut.result()
        APPS[precision] = app
        ENDPOINTS[precision] = endpoint
        print(f"Deployed {precision}: {APP_NAMES[precision]}")


for p in APP_NAMES:
    print(f"  {p} endpoint: {ENDPOINTS[p].web_url}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: each endpoint POST returns 200 with text + gpu_memory_mb.


def checkpoint_1(session):
    for precision, endpoint in ENDPOINTS.items():
        url = endpoint.web_url
        try:
            r = requests.post(url, json={"prompt": "Reply with one word: ok", "max_tokens": 8}, timeout=300)
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"{precision}: POST raised {exc!r}. Cold-load can take 3+ minutes.",
            )
        if r.status_code != 200:
            return CheckpointResult(
                status="fail",
                error_reason=f"{precision}: status {r.status_code}: {r.text[:200]}",
            )
        try:
            body = r.json()
        except Exception:
            return CheckpointResult(status="fail", error_reason=f"{precision}: body not JSON.")
        if "text" not in body or "gpu_memory_mb" not in body:
            return CheckpointResult(
                status="fail",
                error_reason=f"{precision}: missing required keys; got {list(body.keys())}.",
            )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Three precisions, three configs. fp16 uses default LLM kwargs; int8 needs `quantization="bitsandbytes"`; int4 needs `quantization="awq"` AND a pre-quantized model id.

</details>

<details><summary>Hint 2 (direction)</summary>

```
if precision == "fp16":
    pass  # defaults
elif precision == "int8":
    kwargs["quantization"] = "bitsandbytes"
    kwargs["load_format"] = "bitsandbytes"
    kwargs["gpu_memory_utilization"] = 0.75
elif precision == "int4":
    kwargs["quantization"] = "awq"
    kwargs["gpu_memory_utilization"] = 0.70
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Three L4 cold-loads in parallel: ~$0.30 (3 * ~5 minutes of warm at $1.10/hr). Cumulative: ~$0.30.

## Task 2: GPU memory in the benchmark JSON

Smoke-test each endpoint with one prompt. Capture the `gpu_memory_mb` value into a per-precision result dict and persist to `/content/quantization_benchmark.json` with the structure `{precision: {"gpu_memory_mb": N, ...}}`. The validator confirms all three are non-null. It does NOT assert fp16 > int8 > int4.

In [ ]:
# Task 2: per-precision memory probe.

results = {p: {} for p in APP_NAMES}

for precision, endpoint in ENDPOINTS.items():
    r = requests.post(endpoint.web_url, json={"prompt": "Reply with one word: ok", "max_tokens": 8}, timeout=180)
    body = r.json()
    # YOUR CODE: assign results[precision]["gpu_memory_mb"] = body["gpu_memory_mb"].
    raise NotImplementedError("Replace with the per-precision memory capture.")
    results[precision]["smoke_text"] = body["text"]

with open(BENCHMARK_PATH, "w") as f:
    json.dump(results, f, indent=1)
print(json.dumps(results, indent=1))

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: each precision has a non-null gpu_memory_mb in the JSON.


def checkpoint_2(session):
    if not os.path.exists(BENCHMARK_PATH):
        return CheckpointResult(status="fail", error_reason=f"{BENCHMARK_PATH} missing.")
    data = json.loads(open(BENCHMARK_PATH).read())
    for p in ("fp16", "int8", "int4"):
        if p not in data or data[p].get("gpu_memory_mb") in (None, 0):
            return CheckpointResult(
                status="fail",
                error_reason=f"{p}: gpu_memory_mb missing or zero. Check torch.cuda.memory_allocated() is called AFTER the model loads.",
            )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

One line per precision: copy `body["gpu_memory_mb"]` into the results dict.

</details>

<details><summary>Hint 2 (direction)</summary>

```
results[precision]["gpu_memory_mb"] = body["gpu_memory_mb"]
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Three smoke calls: pennies. Cumulative: ~$0.35.

## Task 3: 50-example eval across all three precisions

For each precision, send the same 50 prompts. Capture per-request latency, count output tokens (proxy for throughput), then ask Claude Sonnet (as judge) whether the response is roughly correct on a yes/no basis. Aggregate into `latency_p50_ms`, `latency_p95_ms`, `tokens_per_second`, `accuracy_judge_score` per precision.

The validator confirms the structure has all four fields populated. It does NOT compare them; that lives in the reflection cell.

In [ ]:
# Task 3: 50-example eval across the three endpoints.

EVAL_PROMPTS = [
    "What is the capital of France?",
    "Define 'idempotent' in one sentence.",
    "Which is larger: 7/8 or 0.85?",
    "What does HTTP 429 mean?",
    "What is the boiling point of water at sea level?",
] * 10  # 50 prompts


JUDGE_RUBRIC = (
    "Reply with the single word 'yes' if the response is factually correct OR a reasonable answer "
    "to the user question, otherwise 'no'."
)


def judge_correct(question, response):
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_SONNET, max_tokens=4, system=JUDGE_RUBRIC,
        messages=[{"role": "user", "content": f"QUESTION: {question}\nRESPONSE: {response}\n\nVerdict:"}],
    )
    return "yes" in resp.content[0].text.strip().lower()


def eval_one(precision):
    endpoint = ENDPOINTS[precision]
    latencies = []
    correct = 0
    output_chars = 0
    elapsed = 0.0
    for prompt in EVAL_PROMPTS:
        t0 = time.perf_counter()
        r = requests.post(endpoint.web_url, json={"prompt": prompt, "max_tokens": 80}, timeout=120)
        wall = time.perf_counter() - t0
        elapsed += wall
        body = r.json() if r.status_code == 200 else {"text": "", "latency_ms": int(wall * 1000)}
        latencies.append(body.get("latency_ms", int(wall * 1000)))
        output_chars += len(body.get("text", ""))
        if judge_correct(prompt, body.get("text", "")):
            correct += 1
    # YOUR CODE: compute p50 = float(np.percentile(latencies, 50)),
    # p95 = float(np.percentile(latencies, 95)),
    # tokens_per_second = (output_chars / 4) / max(0.001, elapsed)  # ~4 chars/token approx,
    # accuracy_judge_score = correct / len(EVAL_PROMPTS).
    raise NotImplementedError("Replace with the per-precision aggregation.")


for precision in APP_NAMES:
    print(f"Evaluating {precision}...")
    results[precision].update(eval_one(precision))

with open(BENCHMARK_PATH, "w") as f:
    json.dump(results, f, indent=1)
print(json.dumps(results, indent=1))

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: each precision has latency_p50_ms, latency_p95_ms, tokens_per_second, accuracy_judge_score populated.


def checkpoint_3(session):
    data = json.loads(open(BENCHMARK_PATH).read())
    required = {"latency_p50_ms", "latency_p95_ms", "tokens_per_second", "accuracy_judge_score"}
    for p in ("fp16", "int8", "int4"):
        missing = required - set(data.get(p, {}).keys())
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"{p}: missing fields {missing}. Confirm eval_one returns a dict with all four keys.",
            )
        if data[p]["tokens_per_second"] <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"{p}: tokens_per_second is non-positive ({data[p]['tokens_per_second']}). Check output_chars accumulation.",
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Four numbers per precision. p50/p95 use numpy percentile with q=50 and q=95. Tokens per second is a rough estimate from output chars (4 chars per token is a decent approximation for English).

</details>

<details><summary>Hint 2 (direction)</summary>

```
return {
    "latency_p50_ms": float(np.percentile(latencies, 50)),
    "latency_p95_ms": float(np.percentile(latencies, 95)),
    "tokens_per_second": (output_chars / 4) / max(0.001, elapsed),
    "accuracy_judge_score": correct / len(EVAL_PROMPTS),
}
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 150 generation calls across 3 endpoints + 150 Sonnet judge calls. Modal ~$1.50, Sonnet judge ~$1.50. Cumulative: ~$3.50.

## Task 4: Persist the three-way comparison metric

Compose the `comparisonMetric` string per the Compare-it convention:

```
fp16: p95=Xms, throughput=Y tok/s, accuracy=Z, memory=N MB.
int8: ...
int4: ...
```

Write it to the JSON under key `comparisonMetric`. The validator confirms the structure has 12 fields across the 3 precisions (4 metrics each) plus the string.

In [ ]:
# Task 4: comparison summary.

def build_comparison_metric(data):
    parts = []
    for p in ("fp16", "int8", "int4"):
        d = data[p]
        parts.append(
            f"{p}: p95={d['latency_p95_ms']:.0f}ms, "
            f"throughput={d['tokens_per_second']:.0f} tok/s, "
            f"accuracy={d['accuracy_judge_score']:.2f}, "
            f"memory={d['gpu_memory_mb']} MB"
        )
    return ". ".join(parts) + "."


# YOUR CODE: load results from BENCHMARK_PATH, set
# results["comparisonMetric"] = build_comparison_metric(results), then
# rewrite the file.
raise NotImplementedError("Replace with the comparison-metric writer.")

print(results["comparisonMetric"])

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: JSON has all 12 metric fields plus the comparisonMetric string.


def checkpoint_4(session):
    data = json.loads(open(BENCHMARK_PATH).read())
    expected_metrics = {"latency_p50_ms", "latency_p95_ms", "tokens_per_second", "accuracy_judge_score"}
    for p in ("fp16", "int8", "int4"):
        missing = expected_metrics - set(data.get(p, {}).keys())
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"{p}: missing {missing}. Re-run Task 3.",
            )
    if "comparisonMetric" not in data or not isinstance(data["comparisonMetric"], str):
        return CheckpointResult(
            status="fail",
            error_reason="comparisonMetric string missing from the JSON.",
        )
    # Confirm all three precision names appear in the string.
    for p in ("fp16", "int8", "int4"):
        if p not in data["comparisonMetric"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"comparisonMetric does not mention {p}.",
            )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two lines: open the file, set the key, write.

</details>

<details><summary>Hint 2 (direction)</summary>

```
results = json.loads(open(BENCHMARK_PATH).read())
results["comparisonMetric"] = build_comparison_metric(results)
with open(BENCHMARK_PATH, "w") as f:
    json.dump(results, f, indent=1)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Free. Cumulative: ~$3.50.

## Cleanup

Stop and delete all three Modal apps + Volumes. Run this NOW. Three forgotten L4s = ~$80/day.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down all three Modal apps + Volumes; drop the local JSON.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "modal_app")
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for fid in failed_ids if any(an in fid for an in APP_NAMES.values()))
standard_destroyed = standard_total - (len(failed_ids) - (critical_total - critical_destroyed))
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, Modal apps may still bill. Resolve IMMEDIATELY.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $4 to $6.** Three L4s in parallel; Sonnet judge across 150 questions; everything else small. Three apps, three Volumes, one local JSON: all cleared.

## Reflection

These are not graded. They are for you.

1. The accuracy drop from fp16 to int4 in your run reflects the precision compromise. The team's product can absorb a few points of accuracy loss but not many. What is the next experiment you would run to recover some accuracy without giving up int4 (consider AWQ-aware fine-tuning, larger calibration set, etc.)?

2. The latency improvement from fp16 to int4 was material. At what production QPS does the per-replica cost savings of int4 cover the engineering cost of running int4 (build pipelines, monitoring, rollback)?

## Exam-Style Practice

**Question 1 (precision selection given constraints):**

A team's chatbot uses fp16 Llama 8B at p95 800ms. They quantize to int4 (AWQ); p95 drops to 350ms but eval accuracy drops 5 points. Which is the right call given the constraint "p95 under 500ms, accuracy within 3 points of fp16"?

A. Ship int4; latency is fine.
B. Stay on fp16; accuracy is fine.
C. Try int8.
D. Fine-tune int4 to recover accuracy.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: int4's 5-point accuracy drop violates the accuracy constraint.
- B is wrong: fp16's 800ms p95 violates the latency constraint.
- C is correct: int8 typically splits the difference (latency near 600ms, accuracy within 2 points); a quick benchmark confirms before shipping.
- D is the right longer-term path but a 5-point recovery is uncertain on a deadline.

</details>

**Question 2 (memory-vs-throughput trade-off):**

int4 uses ~6GB on an 8B model where fp16 uses ~16GB. The team asks if they can run 2x int4 endpoints on the same GPU. What is the right consideration?

A. Memory is the only constraint; yes.
B. vLLM's KV cache needs headroom; reducing model memory does not always allow doubling concurrency.
C. Throughput doubles when memory halves.
D. Quantization is reversible at runtime.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: KV cache scales with concurrency and sequence length; you cannot just fill the freed memory with another model.
- B is correct: vLLM's per-request KV cache is the throughput floor; the math is more involved than "we have spare GB".
- C is wrong: throughput is bounded by GPU compute, not just memory.
- D is wrong: quantization is a load-time choice.

</details>

**Question 3 (Compare-it discipline):**

A reviewer asks: "Your benchmark says int4 wins on throughput but loses on accuracy. Why doesn't the checkpoint validator flag the latency/accuracy trade-off automatically?"

A. The validator is buggy.
B. Compare-it checkpoints validate that BOTH architectures exist and emit a metric; they never pick a winner because the choice depends on the constraint scenario.
C. The validator only checks fp16.
D. The judge is unreliable.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: the validator is doing its job.
- B is correct: Compare-it discipline (LAB-CREATION-BLUEPRINT Section 21) is explicit; validators check infrastructure, the reflection cell carries the trade-off cognition.
- C is wrong: the validator checks all three precisions.
- D is unrelated.

</details>